In [1]:
from astropy.io import fits

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from scipy import stats as ss
from scipy.io import readsav
from sklearn.decomposition import PCA

from tqdm import tqdm

import tools as t

## Load data

In [2]:
# load files
path_data = '../data/sim_series/'  # directory where the files are saved
N_files = 20  # 20 temporal series (from 0 to 19)
data_spectra = []
data_rv = []
for i in range(N_files):  # files have the same name, only number changes
    data_spectra.append(fits.getdata(path_data + f'flux_norm_6173_lionel_tau1_spatialscaling1_sansrotation_serie{i}.fits'))
    data_rv.append(readsav(path_data + f'res_rv_bis_6173_lionel_tau1_scaling1_sansrot_serie{i}.sav'))
data_spectra = np.array(data_spectra)
data_rv = np.array(data_rv)
print(data_rv[0].keys())

dict_keys(['rv_gauss', 'rv_correl', 'rv_deriv', 'xbiss', 'ybiss', 'l_true', 'lambd', 'flux_moy'])


In [3]:
c = 3e8  # speed of light [m/s]
# extract data
wl = np.array(data_rv[0]['lambd'])  # same wavelength grid for all time series (verified)
times = np.arange(0, len(data_spectra[0]) * 144, 144) / 60 / 24  # time in days, same for all time series
times_sec = times * 86400  # time in sec
# extract RV data
wl = np.array(wl)

## Work with only one TS

In [8]:
# selecting TS
ts_index = 0

spectra_ex = data_spectra[ts_index]

# splitting TS in two halves
N = len(times)
N_half = N // 2

spectra_0 = spectra_ex[:N_half]
spectra_1 = spectra_ex[N_half:]

## Preprocessing (DO ONLY FOR spectra_0 and spectra_1)

In [9]:
# calculate the mean spectra
mean_spec_0 = np.mean(spectra_0, axis=0)
mean_spec_1 = np.mean(spectra_1, axis=0)

d_S0 = np.gradient(mean_spec_0, wl)
d_S1 = np.gradient(mean_spec_1, wl)

In [10]:
Sf_0, D_0 = t.subtract_D(wl, spectra_0, c=c)  # S_f and Doppler vector
Sf_1, D_1 = t.subtract_D(wl, spectra_1, c=c)

spectra_centered_0 = t.double_centering(Sf_0)
spectra_centered_1 = t.double_centering(Sf_1)

## Calculate RV by template matching

In [12]:
# calculate radial velocity by template matching
C_0 = 1000  # continuum constant

# list of RVs for each timestep
rv_0 = np.array([
    t.calculate_rv(wl, mean_spec_0, spectra_0[i], C_0) for i in range(N_half)
    ])

rv_1 = np.array([
    t.calculate_rv(wl, mean_spec_1, spectra_1[i], C_0) for i in range(N_half)
    ])

## PCA

In [17]:
# apply PCA to the centered matrix
n_components = 16

# first half
pca_flux_0 = PCA(n_components=n_components)           # create PCA object
pcs_0 = pca_flux_0.fit_transform(spectra_centered_0)  # fit the model with spectra_centered and apply the dimensionality reduction
loadings_0 = pca_flux_0.components_                   # save loadings
scores_0 = pcs_0.T                                    # save scores
var_0 = pca_flux_0.explained_variance_ratio_          # save PC variation

# second half
pca_flux_1 = PCA(n_components=n_components)           # create PCA object
pcs_1 = pca_flux_1.fit_transform(spectra_centered_1)  # fit the model with spectra_centered and apply the dimensionality reduction
loadings_1 = pca_flux_1.components_                   # save loadings
scores_1 = pcs_1.T                                    # save scores
var_1 = pca_flux_1.explained_variance_ratio_          # save PC variation